En una planta automotriz, las fallas imprevistas en las prensas hidr´aulicas cr´ıticas generan detenciones en la línea que cuestan miles de dólares por minuto. El departamento de Mantenimiento y
Confiabilidad ha instrumentado una de las máquinas principales para capturar métricas clave.El dataset contiene 1,000 registros históricos y ha sido perfectamente balanceado (500 casos estables y 500 eventos de fallo crítico) para estructurar un modelo predictivo base sin sesgo inicial de clase. A continuación, la siguiente tabla muestra las variables registradas (pestaña Clasificacion Mantenimiento).

a) Entrene un modelo XGBoost realizando una validación cruzada con K = 5. Justifique la elección de hiperparámetros.

b) Extraiga y documente el promedio y la desviación estándar de las siguientes métricas evaluadas a través de las 5 iteraciones: Accuracy, Precision, Recall y F1-Score.

c) Interpretación Industrial Relativa: Si bien el dataset es balanceado para este entrenamiento, en la operación real de la planta las fallas corresponden a menos del 2% del tiempo total. Explique de forma crítica por qué una métrica de Exactitud alta no es suficiente en entornos productivos industriales reales. Asimismo, determine fundadamente en qué métrica específica debe enfocarse el líder de planta si su prioridad estratégica es minimizar los paros no programados (evitar Falsos Negativos), asumiendo el costo menor de inspeccionar falsas alarmas.

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import openpyxl 
from sklearn import metrics
import xgboost as xgb


ruta_exel = r'C:\Users\Vanhertz\Downloads\datasets_evaluacion3.xlsx'
datos = pd.read_excel(ruta_exel, header=0)
datos.describe()


,temperatura_celsius,vibracion_mms,presion_hidraulica_psi,horas_desde_mantenimiento,eficiencia_termica_pct,estado_critico
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000
mean,80.534679,3.331176,31.543628,581.218000,88.241583,0.50000
std,10.119508,1.564654,7.589883,441.580593,8.094319,0.50025
min,59.183184,0.956090,13.368095,30.000000,74.029071,0.00000
25%,71.549018,1.910023,24.677467,172.000000,80.410127,0.00000
50%,79.374305,2.757684,33.261955,459.000000,90.493717,0.50000
75%,89.669757,4.721504,38.451356,997.000000,95.659662,1.00000
max,106.304552,8.137302,43.663535,1399.000000,98.488015,1.00000


**Crear y hacer splitting de X e Y**

In [7]:
#crear X,Y
X = datos.iloc[:, [0,1,2,4]].values # se sacan los entereos pues solo podemos usar variables continuas
y = datos.iloc[:, 5].values

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.30,stratify=y,random_state=1) 
#siempre lo mismo como en las otras clases.
#para crear conjuntos de entrenamiento y prueba
#X

In [8]:
#entrenamos el modelo con XGBoost usando make pipe
from sklearn.pipeline import make_pipeline               #es necesario para hacer un pipeline, que es una forma de concatenar proceso
pipe_xgb = make_pipeline(StandardScaler(), xgb.XGBClassifier(n_estimators=3, max_depth=3, learning_rate=0.05, objective='binary:logistic'))
pipe_xgb.fit(X_train, y_train)      #con esto entrenamos el modelo con los datos de entrenamiento

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('xgbclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None


In [9]:
# Validacion cruzada StratifiedKFold
#se tiene que estandarizar porque los valores de las variables son muy diferentes, es para que no tengan un peso mayor en el modelo
#valores entre 59 y 106 en grados y 0.09 a 8.13 en vibracion, si o si hay que estandarizar.
#en este caso por comodidad usamos los codigos de la clase 18 para crear un pipeline

from sklearn.model_selection import StratifiedKFold       

kfold = StratifiedKFold(n_splits=5).split(X_train, y_train)
model_xgb = xgb.XGBClassifier(n_estimators=2, max_depth=3, learning_rate=0.01, objective='binary:logistic')

scores = []
for k, (train, test) in enumerate(kfold):
    pipe_temp = make_pipeline(StandardScaler(),model_xgb)     # se cambio logistic regression por xgboost y lr por temp(pues sera solo dentro del for)
    pipe_temp.fit(X_train[train], y_train[train])
    score = pipe_temp.score(X_train[test], y_train[test])
    scores.append(score)
    print('Fold: %2d, Class dist train.: %s, Class dist test.: %s, Acc: %.3f' % (k+1,np.bincount(y_train[train]),
                                                                                 np.bincount(y_train[test]), score))
    
print('\nCV accuracy: %.3f +/- %.3f' % (np.mean(scores), np.std(scores)))

Fold:  1, Class dist train.: [280 280], Class dist test.: [70 70], Acc: 1.000
Fold:  2, Class dist train.: [280 280], Class dist test.: [70 70], Acc: 1.000
Fold:  3, Class dist train.: [280 280], Class dist test.: [70 70], Acc: 1.000
Fold:  4, Class dist train.: [280 280], Class dist test.: [70 70], Acc: 1.000
Fold:  5, Class dist train.: [280 280], Class dist test.: [70 70], Acc: 0.986

CV accuracy: 0.997 +/- 0.006


In [10]:
#para evaluar el modelo se debe usar el codigo de la clase 18 tambien:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from sklearn.metrics import matthews_corrcoef,cohen_kappa_score
from sklearn.metrics import confusion_matrix

y_pred = pipe_xgb.predict(X_test)

confmat = confusion_matrix(y_true=y_test, y_pred=y_pred)
print(confmat)

print('Accuracy (ec. 2): %.3f' % accuracy_score(y_true=y_test, y_pred=y_pred))
print('Precision (ec. 5): %.3f' % precision_score(y_true=y_test, y_pred=y_pred))
print('Recall (ec. 6): %.3f' % recall_score(y_true=y_test, y_pred=y_pred))
print('F1 (ec. 9): %.3f' % f1_score(y_true=y_test, y_pred=y_pred))
print('MCC (ec. 10): %.3f' % matthews_corrcoef(y_true=y_test, y_pred=y_pred))
print('Kappa (ec. 11): %.3f' % cohen_kappa_score(y1=y_test, y2=y_pred))


[[150   0]
 [  0 150]]
Accuracy (ec. 2): 1.000
Precision (ec. 5): 1.000
Recall (ec. 6): 1.000
F1 (ec. 9): 1.000
MCC (ec. 10): 1.000
Kappa (ec. 11): 1.000
